In [ ]:
# @title 0) 저장소 클론·pip 설치 (Colab에서만 사용, 로컬에서는 생략 가능)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
TARGET = Path("/content/colab")

if Path("/content").exists() and not (Path.cwd() / "src" / "mindscopex_analysis").exists():
    if not TARGET.exists():
        print(f"+ git clone --depth 1 {REPO_URL} {TARGET}")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(TARGET)])
    os.chdir(TARGET)
    os.environ["MINDSCOPEX_ROOT"] = str(TARGET)
    print("cwd =", Path.cwd())
    print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[dev]"])
else:
    print("로컬 저장소에서 실행 중입니다. 설치 셀은 건너뜁니다.")


# Qwen-Scope 기반 직관 함정(lure) feature 연구

이 노트북은 이미지의 연구 질문을 한 번에 끝내기보다, RQ1→RQ5 순서로 산출물을 쌓아가는 작업 파일입니다.

- **RQ1** Base 모델에서 lure를 만드는 feature 집합을 찾을 수 있는가?
- **RQ2** Reasoning on/off에서 그 feature는 삭제, 억제, 우회 중 무엇에 가까운가?
- **RQ3** 그 처리가 정답 생성에 인과적으로 필요한가?
- **RQ4** 갈등 감지와 정답 계산은 분리된 회로인가?
- **RQ5** 이 mechanism은 다른 직관 함정 과제와 얼마나 공유되는가?

기본 설정은 `Qwen/Qwen3-1.7B-Base` + `Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50`이고, reasoning on/off는 `Qwen/Qwen3-1.7B`의 `enable_thinking` 스위치로 비교합니다.


## 1. Import와 설정 로드


In [ ]:
import gc
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import torch
import yaml
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

_REPO_MARK = Path("src") / "mindscopex_analysis" / "__init__.py"


def _find_repo_root() -> Path:
    env = os.environ.get("MINDSCOPEX_ROOT", "").strip()
    if env:
        root = Path(env).expanduser().resolve()
        if (root / _REPO_MARK).is_file():
            return root
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / _REPO_MARK).is_file():
            return base
    raise FileNotFoundError("src/mindscopex_analysis 를 찾지 못했습니다.")

ROOT = _find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mindscopex_analysis.notebook_utils import dtype_from_str, load_notebook_experiment_config
from mindscopex_analysis.qwen_scope import (
    capture_residuals,
    format_qwen_chat,
    get_transformer_layers,
    load_qwen_scope_sae,
    make_feature_steering_hook,
    split_qwen_thinking,
    summarize_qwen_scope_features,
)

pio.renderers.default = "plotly_mimetype"
set_seed(42)
print("ROOT =", ROOT)
print("torch =", torch.__version__, "cuda =", torch.cuda.is_available())


In [ ]:
YAML_PATH = ROOT / "configs" / "qwen_scope_lure_research.yaml"
EXP = load_notebook_experiment_config(ROOT, YAML_PATH)

RUNTIME = EXP["runtime"]
QSC = EXP["qwen_scope"]
GEN = EXP["generation"]
ANALYSIS = EXP["analysis"]
TASKS = EXP["tasks"]

DEVICE = RUNTIME.get("device") or ("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = dtype_from_str(str(RUNTIME.get("dtype", "bfloat16")))
TRUST_REMOTE_CODE = bool(RUNTIME.get("trust_remote_code", True))
LAYERS = [int(x) for x in QSC["layers"]]
MAX_LENGTH = int(RUNTIME.get("max_length", 2048))
TOKEN_POSITION = str(QSC.get("token_position", "last"))
BATCH_SIZE = int(QSC.get("batch_size", 64))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
OUT_DIR = ROOT / "outputs" / "qwen_scope_lure_research" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"experiment = {EXP['experiment_name']}")
print(f"device={DEVICE} dtype={DTYPE} layers={LAYERS} token_position={TOKEN_POSITION}")
print("base model:", EXP["models"]["base"])
print("chat model:", EXP["models"]["chat"])
print("qwen-scope:", QSC["repo_id"])
print("out:", OUT_DIR)

task_df = pd.DataFrame(TASKS)
required_cols = {"id", "family", "matched_pair", "language", "is_lure", "lure_type", "prompt"}
missing_cols = sorted(required_cols.difference(task_df.columns))
if missing_cols:
    raise ValueError(f"데이터셋 필수 컬럼 누락: {missing_cols}")

pair_check = (
    task_df.groupby("matched_pair")["is_lure"]
    .agg(n="count", n_lure="sum")
    .reset_index()
)
bad_pairs = pair_check[(pair_check["n"] != 2) | (pair_check["n_lure"] != 1)]
if not bad_pairs.empty:
    display(bad_pairs)
    raise ValueError("각 matched_pair는 정확히 lure 1개와 control 1개여야 합니다.")

task_df["prompt_chars"] = task_df["prompt"].map(lambda x: len(str(x)))
family_balance = task_df.groupby(["family", "is_lure"]).size().unstack(fill_value=0)
language_balance = task_df.groupby(["language", "is_lure"]).size().unstack(fill_value=0)

print(f"tasks={len(task_df)}  matched_pairs={task_df['matched_pair'].nunique()}  lure={int(task_df['is_lure'].sum())}  control={int((~task_df['is_lure']).sum())}")
display(family_balance)
display(language_balance)
display(task_df[["id", "matched_pair", "family", "language", "is_lure", "lure_type", "prompt_chars", "prompt"]])


In [ ]:
def verify_answer(text: str, expected_regex: str | None, lure_regex: str | None = None) -> dict:
    expected_regex = expected_regex or ""
    lure_regex = lure_regex or ""
    is_correct = bool(expected_regex and re.search(expected_regex, text, flags=re.DOTALL))
    hit_lure = bool(lure_regex and re.search(lure_regex, text, flags=re.DOTALL))
    if is_correct:
        status = "correct"
    elif hit_lure:
        status = "lure"
    else:
        status = "other"
    return {"answer_status": status, "is_correct": is_correct, "hit_lure": hit_lure}


def generation_kwargs_for(mode: str) -> dict:
    do_sample = bool(GEN.get("do_sample", True))
    kwargs = {
        "max_new_tokens": int(GEN.get("max_new_tokens", 1024)),
        "do_sample": do_sample,
    }
    if do_sample:
        if mode == "think_on":
            kwargs.update({"temperature": 0.6, "top_p": 0.95, "top_k": 20})
        else:
            kwargs.update({
                "temperature": float(GEN.get("temperature", 0.7)),
                "top_p": float(GEN.get("top_p", 0.8)),
                "top_k": 20,
            })
    return kwargs


def top_feature_rows(layer: int, label: str, vector: np.ndarray, top_n: int) -> list[dict]:
    idx = np.argsort(np.abs(vector))[::-1][:top_n]
    return [
        {"layer": layer, "rank": i + 1, "feature": int(f), label: float(vector[f])}
        for i, f in enumerate(idx)
    ]


## 2. 답변 생성: Qwen3 reasoning on/off 행동 비교

여기서는 외부 행동을 먼저 확인합니다. `think_off`와 `think_on` 모두 같은 문제를 풀고, 정답/직관 오답/기타로 느슨하게 분류합니다.


In [ ]:
CHAT_MODEL_ID = EXP["models"]["chat"]
SYSTEM_PROMPT = str(GEN.get("system_prompt") or "")
MODES = list(GEN.get("modes") or ["think_off", "think_on"])

tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=TRUST_REMOTE_CODE,
).to(DEVICE).eval()
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

rows = []
for mode in MODES:
    enable_thinking = mode == "think_on"
    for task in tqdm(TASKS, desc=f"generate {mode}"):
        formatted = format_qwen_chat(
            tokenizer,
            task["prompt"],
            system_prompt=SYSTEM_PROMPT,
            enable_thinking=enable_thinking,
        )
        inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        gen_kwargs = generation_kwargs_for(mode)
        gen_kwargs.update({"pad_token_id": tokenizer.pad_token_id, "eos_token_id": tokenizer.eos_token_id})
        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)
        new_tokens = out[0, inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(new_tokens, skip_special_tokens=False).strip()
        thinking, final_answer = split_qwen_thinking(raw)
        judged = verify_answer(final_answer or raw, task.get("expected_regex"), task.get("lure_regex"))
        rows.append({
            "mode": mode,
            "task_id": task["id"],
            "family": task["family"],
            "is_lure": bool(task["is_lure"]),
            "lure_type": task["lure_type"],
            "prompt": task["prompt"],
            "raw_generation": raw,
            "thinking": thinking,
            "answer": final_answer or raw,
            "thinking_tokens_approx": len(thinking.split()),
            **judged,
        })

generation_df = pd.DataFrame(rows)
generation_df.to_csv(OUT_DIR / "generation_reasoning_on_off.csv", index=False, encoding="utf-8-sig")
display(generation_df[["mode", "task_id", "answer_status", "is_correct", "hit_lure", "thinking_tokens_approx", "answer"]])

summary = generation_df.groupby(["mode", "is_lure", "answer_status"]).size().reset_index(name="n")
display(summary)

# VRAM 절약: 다음 단계에서 필요하면 다시 로드합니다.
del model, tokenizer
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()


## 3. RQ1: Base 모델에서 lure feature 후보 찾기

가설: Base 모델의 residual stream에는 직관 함정 과제에서 control 과제보다 더 강하게 켜지는 SAE feature 집합이 있다. 여기서는 각 layer마다 `mean(lure) - mean(control)`가 큰 feature를 후보로 잡습니다.


In [ ]:
BASE_MODEL_ID = EXP["models"]["base"]
SAE_REPO = QSC["repo_id"]
SAE_TOP_K = int(QSC.get("top_k", 50))
REPORT_TOP_N = int(ANALYSIS.get("report_top_n", 25))
CANDIDATE_TOP_N = int(ANALYSIS.get("candidate_top_n", 50))

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=TRUST_REMOTE_CODE,
).to(DEVICE).eval()

feature_rows = []
feature_matrices = {}

# Prompt별 residual은 작게 잡기 위해 token_position=last가 기본입니다.
residual_by_task = {}
for task in tqdm(TASKS, desc="capture base residuals"):
    residual_by_task[task["id"]] = capture_residuals(
        base_model,
        base_tokenizer,
        [task["prompt"]],
        LAYERS,
        device=DEVICE,
        max_length=MAX_LENGTH,
        token_position=TOKEN_POSITION,
    )

for layer in tqdm(LAYERS, desc="Qwen-Scope feature summary"):
    sae = load_qwen_scope_sae(
        SAE_REPO,
        layer,
        device=DEVICE,
        dtype=DTYPE,
        top_k=SAE_TOP_K,
    )
    mat = []
    labels = []
    for task in TASKS:
        summary = summarize_qwen_scope_features(
            residual_by_task[task["id"]][layer],
            sae,
            batch_size=BATCH_SIZE,
        )
        mean_vec = summary["mean"].numpy()
        mat.append(mean_vec)
        labels.append(task["id"])
        top_idx = np.argsort(mean_vec)[::-1][:10]
        for rank, feature in enumerate(top_idx, start=1):
            feature_rows.append({
                "layer": layer,
                "task_id": task["id"],
                "family": task["family"],
                "is_lure": bool(task["is_lure"]),
                "lure_type": task["lure_type"],
                "feature": int(feature),
                "rank_in_task": rank,
                "mean_activation": float(mean_vec[feature]),
                "n_tokens": int(summary["n_tokens"]),
            })
    feature_matrices[layer] = {"task_ids": labels, "matrix": np.stack(mat)}
    del sae
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

feature_rows_df = pd.DataFrame(feature_rows)
feature_rows_df.to_csv(OUT_DIR / "rq1_task_top_features.csv", index=False, encoding="utf-8-sig")

del base_model, base_tokenizer, residual_by_task
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("feature_matrices layers:", list(feature_matrices))
display(feature_rows_df.head(20))


In [ ]:
candidate_rows = []
labels_by_id = {task["id"]: bool(task["is_lure"]) for task in TASKS}
meta_by_id = {task["id"]: task for task in TASKS}
pairs = sorted({task["matched_pair"] for task in TASKS})

for layer, bundle in feature_matrices.items():
    task_ids = bundle["task_ids"]
    X = bundle["matrix"]
    row_by_id = {tid: i for i, tid in enumerate(task_ids)}
    y = np.array([labels_by_id[t] for t in task_ids], dtype=bool)
    lure_mean = X[y].mean(axis=0)
    control_mean = X[~y].mean(axis=0)
    global_delta = lure_mean - control_mean
    global_effect = global_delta / (X.std(axis=0) + 1e-8)

    pair_deltas = []
    for pair in pairs:
        pair_tasks = [t for t in TASKS if t["matched_pair"] == pair]
        lure_id = next(t["id"] for t in pair_tasks if t["is_lure"])
        control_id = next(t["id"] for t in pair_tasks if not t["is_lure"])
        pair_deltas.append(X[row_by_id[lure_id]] - X[row_by_id[control_id]])
    pair_deltas = np.stack(pair_deltas)
    pair_delta_mean = pair_deltas.mean(axis=0)
    pair_delta_std = pair_deltas.std(axis=0, ddof=1) + 1e-8
    paired_effect = pair_delta_mean / pair_delta_std
    paired_consistency = (np.sign(pair_deltas) == np.sign(pair_delta_mean)).mean(axis=0)

    top = np.argsort(np.abs(paired_effect))[::-1][:CANDIDATE_TOP_N]
    for rank, feature in enumerate(top, start=1):
        candidate_rows.append({
            "layer": layer,
            "rank": rank,
            "feature": int(feature),
            "lure_mean": float(lure_mean[feature]),
            "control_mean": float(control_mean[feature]),
            "delta_lure_minus_control": float(global_delta[feature]),
            "effect_size": float(global_effect[feature]),
            "paired_delta_mean": float(pair_delta_mean[feature]),
            "paired_effect_size": float(paired_effect[feature]),
            "paired_consistency": float(paired_consistency[feature]),
            "abs_effect_size": float(abs(paired_effect[feature])),
        })

candidate_features = pd.DataFrame(candidate_rows)
candidate_features.to_csv(OUT_DIR / "rq1_candidate_lure_features.csv", index=False, encoding="utf-8-sig")
display(candidate_features.head(REPORT_TOP_N))

fig = go.Figure()
for layer, sdf in candidate_features.groupby("layer"):
    fig.add_trace(go.Bar(
        x=[f"L{layer}/F{f}" for f in sdf.head(10)["feature"]],
        y=sdf.head(10)["paired_effect_size"],
        name=f"layer {layer}",
    ))
fig.update_layout(
    title="RQ1 후보 lure features: matched-pair effect size 상위",
    xaxis_title="Layer / SAE feature",
    yaxis_title="mean paired delta / std paired delta",
    template="plotly_white",
    height=450,
)
fig.show()


## 4. RQ2: Reasoning on/off에서 lure feature는 어떻게 처리되는가?

가설 A: reasoning on은 lure feature 자체를 낮춘다.  
가설 B: lure feature는 유지되지만, 이후 layer나 별도 feature로 우회/보정된다.  
가설 C: lure feature는 높아지고, 갈등 감지 신호처럼 쓰인다.

아래 셀은 같은 Qwen-Scope feature 후보를 chat 모델의 `think_off`/`think_on` formatted prompt에 투영합니다. Qwen-Scope는 Base SAE지만, 공개 모델 카드에서도 post-training checkpoint 분석에 대체로 쓸 수 있다고 안내합니다.


In [ ]:
chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
chat_model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=TRUST_REMOTE_CODE,
).to(DEVICE).eval()

mode_residuals = {}
for mode in MODES:
    enable_thinking = mode == "think_on"
    for task in tqdm(TASKS, desc=f"capture {mode}"):
        text = format_qwen_chat(
            chat_tokenizer,
            task["prompt"],
            system_prompt=SYSTEM_PROMPT,
            enable_thinking=enable_thinking,
        )
        mode_residuals[(mode, task["id"])] = capture_residuals(
            chat_model,
            chat_tokenizer,
            [text],
            LAYERS,
            device=DEVICE,
            max_length=MAX_LENGTH,
            token_position=TOKEN_POSITION,
        )

mode_feature_rows = []
selected_by_layer = {
    layer: candidate_features[candidate_features["layer"] == layer].head(CANDIDATE_TOP_N)["feature"].astype(int).tolist()
    for layer in LAYERS
}

for layer in tqdm(LAYERS, desc="project candidate features"):
    sae = load_qwen_scope_sae(SAE_REPO, layer, device=DEVICE, dtype=DTYPE, top_k=SAE_TOP_K)
    for mode in MODES:
        for task in TASKS:
            summary = summarize_qwen_scope_features(
                mode_residuals[(mode, task["id"])][layer],
                sae,
                batch_size=BATCH_SIZE,
            )
            mean_vec = summary["mean"].numpy()
            for feature in selected_by_layer[layer]:
                mode_feature_rows.append({
                    "mode": mode,
                    "layer": layer,
                    "feature": int(feature),
                    "task_id": task["id"],
                    "family": task["family"],
                    "is_lure": bool(task["is_lure"]),
                    "lure_type": task["lure_type"],
                    "mean_activation": float(mean_vec[feature]),
                })
    del sae
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

mode_features = pd.DataFrame(mode_feature_rows)
mode_features.to_csv(OUT_DIR / "rq2_reasoning_mode_candidate_features.csv", index=False, encoding="utf-8-sig")

del chat_model, chat_tokenizer, mode_residuals
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

display(mode_features.head(20))


In [ ]:
rq2 = (
    mode_features
    .groupby(["mode", "layer", "feature", "is_lure"], as_index=False)["mean_activation"]
    .mean()
)
rq2_pivot = rq2.pivot_table(
    index=["layer", "feature", "is_lure"],
    columns="mode",
    values="mean_activation",
).reset_index()
if {"think_on", "think_off"}.issubset(rq2_pivot.columns):
    rq2_pivot["think_on_minus_off"] = rq2_pivot["think_on"] - rq2_pivot["think_off"]
else:
    rq2_pivot["think_on_minus_off"] = np.nan

rq2_lure = rq2_pivot[rq2_pivot["is_lure"]].copy()
rq2_lure = rq2_lure.merge(
    candidate_features[["layer", "feature", "effect_size", "paired_effect_size", "paired_consistency"]],
    on=["layer", "feature"],
    how="left",
)
rq2_lure = rq2_lure.sort_values("paired_effect_size", key=lambda s: s.abs(), ascending=False)
display(rq2_lure.head(REPORT_TOP_N))

fig = go.Figure()
for layer, sdf in rq2_lure.groupby("layer"):
    top = sdf.head(12)
    fig.add_trace(go.Bar(
        x=[f"L{layer}/F{int(f)}" for f in top["feature"]],
        y=top["think_on_minus_off"],
        name=f"layer {layer}",
    ))
fig.update_layout(
    title="RQ2: candidate lure features의 think_on - think_off 변화(lure tasks)",
    xaxis_title="Layer / feature",
    yaxis_title="? mean activation",
    template="plotly_white",
    height=450,
)
fig.show()


## 5. RQ3: feature 처리가 정답 생성에 인과적으로 필요한가?

첫 패스에서는 후보 feature의 decoder 방향을 residual stream에 더하거나 빼서 행동 변화를 봅니다. 이 셀은 비용이 커서 기본값은 꺼져 있습니다. RQ1/RQ2 결과를 확인한 뒤 `RUN_RQ3_INTERVENTIONS = True`로 바꿔 실행하세요.


In [ ]:
RUN_RQ3_INTERVENTIONS = False

if not RUN_RQ3_INTERVENTIONS:
    print("RQ3 intervention은 아직 실행하지 않았습니다. RUN_RQ3_INTERVENTIONS=True 로 바꿔 실행하세요.")
else:
    chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
    chat_model = AutoModelForCausalLM.from_pretrained(
        CHAT_MODEL_ID,
        torch_dtype=DTYPE,
        trust_remote_code=TRUST_REMOTE_CODE,
    ).to(DEVICE).eval()
    if chat_tokenizer.pad_token_id is None and chat_tokenizer.eos_token_id is not None:
        chat_tokenizer.pad_token = chat_tokenizer.eos_token

    target_row = candidate_features.iloc[0]
    target_layer = int(target_row["layer"])
    n_features = int(ANALYSIS.get("rq3_default_feature_count", 8))
    target_features = candidate_features[candidate_features["layer"] == target_layer].head(n_features)["feature"].astype(int).tolist()
    sae = load_qwen_scope_sae(SAE_REPO, target_layer, device=DEVICE, dtype=DTYPE, top_k=SAE_TOP_K)
    blocks = get_transformer_layers(chat_model)

    intervention_rows = []
    coeffs = list(ANALYSIS.get("steering_coefficients", [-4.0, 4.0]))
    lure_tasks = [t for t in TASKS if t["is_lure"]]
    for coeff in coeffs:
        handle = blocks[target_layer].register_forward_hook(
            make_feature_steering_hook(sae, target_features, coefficient=float(coeff), token_position="last")
        )
        try:
            for task in tqdm(lure_tasks, desc=f"steer coeff={coeff}"):
                text = format_qwen_chat(
                    chat_tokenizer,
                    task["prompt"],
                    system_prompt=SYSTEM_PROMPT,
                    enable_thinking=False,
                )
                inputs = chat_tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
                inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
                gen_kwargs = generation_kwargs_for("think_off")
                gen_kwargs.update({"pad_token_id": chat_tokenizer.pad_token_id, "eos_token_id": chat_tokenizer.eos_token_id})
                with torch.no_grad():
                    out = chat_model.generate(**inputs, **gen_kwargs)
                new_tokens = out[0, inputs["input_ids"].shape[1]:]
                raw = chat_tokenizer.decode(new_tokens, skip_special_tokens=False).strip()
                thinking, answer = split_qwen_thinking(raw)
                judged = verify_answer(answer or raw, task.get("expected_regex"), task.get("lure_regex"))
                intervention_rows.append({
                    "coeff": coeff,
                    "layer": target_layer,
                    "features": ",".join(map(str, target_features)),
                    "task_id": task["id"],
                    "answer": answer or raw,
                    **judged,
                })
        finally:
            handle.remove()

    intervention_df = pd.DataFrame(intervention_rows)
    intervention_df.to_csv(OUT_DIR / "rq3_feature_steering.csv", index=False, encoding="utf-8-sig")
    display(intervention_df[["coeff", "task_id", "answer_status", "answer"]])

    del chat_model, chat_tokenizer, sae
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


## 6. RQ4: 갈등 감지와 정답 계산 회로 분리성

작은 실험에서는 완전한 회로 복원 대신 세 가지 관찰량을 봅니다.

1. Base lure effect가 어느 layer에서 커지는가?
2. reasoning on/off 차이가 같은 feature에서 일어나는가, 다른 feature에서 일어나는가?
3. 행동 정답률과 candidate feature score가 같은 방향으로 움직이는가?


In [ ]:
# layer별 lure effect 절댓값 평균
rq4_layer = candidate_features.groupby("layer", as_index=False).agg(
    lure_effect_abs_mean=("abs_effect_size", "mean"),
    lure_effect_max=("abs_effect_size", "max"),
)

if "rq2_lure" in globals() and not rq2_lure.empty:
    rq2_layer = rq2_lure.groupby("layer", as_index=False).agg(
        reasoning_shift_mean=("think_on_minus_off", "mean"),
        reasoning_shift_abs_mean=("think_on_minus_off", lambda s: float(np.mean(np.abs(s)))),
    )
    rq4_layer = rq4_layer.merge(rq2_layer, on="layer", how="left")

# 후보 feature score와 행동 결과 연결
score_rows = []
for _, r in mode_features.iterrows():
    score_rows.append(r.to_dict())
score_df = pd.DataFrame(score_rows)
score_summary = score_df.groupby(["mode", "task_id", "layer"], as_index=False)["mean_activation"].mean()
behavior = generation_df[["mode", "task_id", "answer_status", "is_correct", "hit_lure"]]
score_behavior = score_summary.merge(behavior, on=["mode", "task_id"], how="left")

correct_assoc = []
for (mode, layer), sdf in score_behavior.groupby(["mode", "layer"]):
    if sdf["is_correct"].nunique() < 2:
        diff = np.nan
    else:
        diff = sdf[sdf["is_correct"]]["mean_activation"].mean() - sdf[~sdf["is_correct"]]["mean_activation"].mean()
    correct_assoc.append({"mode": mode, "layer": layer, "correct_minus_incorrect_score": diff})
correct_assoc = pd.DataFrame(correct_assoc)

rq4_layer.to_csv(OUT_DIR / "rq4_layer_separation_metrics.csv", index=False, encoding="utf-8-sig")
correct_assoc.to_csv(OUT_DIR / "rq4_correctness_association.csv", index=False, encoding="utf-8-sig")
display(rq4_layer)
display(correct_assoc)

fig = go.Figure()
fig.add_trace(go.Scatter(x=rq4_layer["layer"], y=rq4_layer["lure_effect_abs_mean"], mode="lines+markers", name="Base lure effect"))
if "reasoning_shift_abs_mean" in rq4_layer.columns:
    fig.add_trace(go.Scatter(x=rq4_layer["layer"], y=rq4_layer["reasoning_shift_abs_mean"], mode="lines+markers", name="Reasoning shift"))
fig.update_layout(
    title="RQ4: layer별 lure 감지 신호와 reasoning 처리 신호",
    xaxis_title="Layer",
    yaxis_title="Mean absolute score",
    template="plotly_white",
)
fig.show()


## 7. RQ5: 다른 직관 함정 과제와 mechanism 공유성

가족별 top feature set의 Jaccard overlap을 봅니다. 표본이 작을 때는 결론이 아니라 다음 데이터 수집 방향을 정하는 탐색 지표로 해석합니다.


In [ ]:
family_feature_sets = {}
for layer, bundle in feature_matrices.items():
    task_ids = bundle["task_ids"]
    X = bundle["matrix"]
    for family in sorted({t["family"] for t in TASKS}):
        idx = [i for i, tid in enumerate(task_ids) if meta_by_id[tid]["family"] == family]
        other = [i for i, tid in enumerate(task_ids) if meta_by_id[tid]["family"] != family]
        if not idx or not other:
            continue
        delta = X[idx].mean(axis=0) - X[other].mean(axis=0)
        top = set(np.argsort(np.abs(delta))[::-1][:REPORT_TOP_N].tolist())
        family_feature_sets[(layer, family)] = top

families = sorted({t["family"] for t in TASKS})
for layer in LAYERS:
    z = np.zeros((len(families), len(families)))
    for i, a in enumerate(families):
        for j, b in enumerate(families):
            A = family_feature_sets.get((layer, a), set())
            B = family_feature_sets.get((layer, b), set())
            z[i, j] = len(A & B) / max(len(A | B), 1)
    fig = go.Figure(data=go.Heatmap(
        z=z,
        x=families,
        y=families,
        colorscale="Blues",
        zmin=0,
        zmax=1,
        text=np.round(z, 2).astype(str),
        texttemplate="%{text}",
    ))
    fig.update_layout(title=f"RQ5: family? top feature Jaccard overlap (Layer {layer})", template="plotly_white")
    fig.show()

rows = []
for (layer, family), feats in family_feature_sets.items():
    rows.append({"layer": layer, "family": family, "features": ",".join(map(str, sorted(feats)))})
rq5_sets = pd.DataFrame(rows)
rq5_sets.to_csv(OUT_DIR / "rq5_family_feature_sets.csv", index=False, encoding="utf-8-sig")
display(rq5_sets)


## 8. 저장물

노트북이 생성하는 주요 파일:

- `generation_reasoning_on_off.csv`: Qwen3 reasoning on/off 답변과 정답/직관오답 판정
- `rq1_candidate_lure_features.csv`: Base 모델 lure feature 후보
- `rq2_reasoning_mode_candidate_features.csv`: 후보 feature의 thinking on/off 변화
- `rq4_layer_separation_metrics.csv`: 갈등 감지/처리 신호의 layer별 분리 탐색
- `rq5_family_feature_sets.csv`: 과제 family별 feature set 공유성

다음 단계에서 RQ3를 더 엄밀히 하려면, feature steering뿐 아니라 activation patching으로 `lure prompt → corrected prompt`, `think_off → think_on` donor/receiver를 나누어 layer별 causal trace를 추가하면 됩니다.
